In [1]:
import numpy as np
import pandas as pd

# 1. 利用 haversine 公式計算兩點於地球上的距離
def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6371.0 
    rad_lat1 = np.radians(lat1)
    rad_lon1 = np.radians(lon1)
    rad_lat2 = np.radians(lat2)
    rad_lon2 = np.radians(lon2)
    
    dlat = rad_lat2 - rad_lat1
    dlon = rad_lon2 - rad_lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(rad_lat1) * np.cos(rad_lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = earth_radius * c
    
    return distance

# 2. 利用地震動衰減公式計算 PGA 值
def calculate_pga(magnitude, distance):
    # 定義經驗常數 (以台灣典型地震衰減模型進行簡化)
    A = 0.026
    B = 1.28
    C = 10.0  # 近震央修正項
    D = 1.5   # 幾何衰減指數

    base_pga = (A * np.exp(B * magnitude)) / ((distance + C) ** D)
    
    # 4. 轉換為氣象署常用的 gal 單位
    final_pga = base_pga * 980
    
    return final_pga

# 3. 將 PGA 轉換為震度級數
def get_intensity_level(pga):
    if pga < 0.8: return "0級"
    elif pga < 2.5: return "1級"
    elif pga < 8.0: return "2級"
    elif pga < 25.0: return "3級"
    elif pga < 80.0: return "4級"
    elif pga < 140.0: return "5弱"
    elif pga < 250.0: return "5強"
    elif pga < 440.0: return "6弱"
    elif pga < 800.0: return "6強"
    else: return "7級"

# 4. 設定老屋在該震度級數的受損機率
def get_damage_rate(intensity): 
    damage_table = {
        "0級": 0.00, "1級": 0.00, "2級": 0.00, "3級": 0.00,
        "4級": 0.01,  
        "5弱": 0.05,  
        "5強": 0.15,
        "6弱": 0.35,
        "6強": 0.60,
        "7級": 0.85
    }
    return damage_table.get(intensity, 0.00)

# 5. 整合上方 function 進行最終計算
def run_simulation(epicenter_lat, epicenter_lon, magnitude, area_df):
    results = []
    for idx, row in area_df.iterrows():
        dist = haversine_distance(epicenter_lat, epicenter_lon, row['lat'], row['lon'])
        pga = calculate_pga(magnitude, dist)
        intensity = get_intensity_level(pga)
        damage_rate = get_damage_rate(intensity)
            
        # 災民預估模型公式： 該里總人口 * 該里老屋比例 * 該震度下的老屋損壞率 * 0.8
        predicted_refugees = int(row['population'] * row['old_house_ratio'] * damage_rate * 0.8)
            
        results.append({
            '行政區': row['行政區'],
            '里名': row['里名'],  
            'lat': row['lat'],
            'lon': row['lon'],
            '震央距離_km': round(dist, 2),
            '預估PGA': round(pga, 2),
            '預估震度': intensity,
            '預估避難人數': predicted_refugees
        })
            
    return pd.DataFrame(results) 


if __name__ == "__main__":
    
    csv_filename = "各里資料.csv" #檔名可修改，需與程式碼放在同一資料夾

    # 1.從外部 CSV 檔案讀取資料 
    try:
        df_areas = pd.read_csv(csv_filename, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"錯誤：找不到檔案 '{csv_filename}'，請確認檔案路徑是否正確。")
        exit()

    # 2.輸入
    try:
        epi_lat = float(input("請輸入震央緯度（例如 24.15）："))
        epi_lon = float(input("請輸入震央經度（例如 121.62）："))
        mag = float(input("請輸入地震規模（例如 6.0）："))
    except ValueError:
        print("錯誤：輸入格式不正確，經緯度與規模必須是數字。")
        exit()

    # 3.執行模擬
    df_output = run_simulation(epi_lat, epi_lon, mag, df_areas)
    
    # 4.輸出
    json_result = df_output.to_json(orient='records', force_ascii=False, indent=4)
    print(json_result)
    
    # 5.匯出成實體 .json 檔案
    output_json_file = "earthquake_simulation_result.json"
    with open(output_json_file, "w", encoding="utf-8") as f:
        f.write(json_result)
        
    print(f"\n[系統提示] JSON 檔案已成功匯出至: {output_json_file}")